In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/fraud-project/'
print("Drive connected!")

Mounted at /content/drive
Drive connected!


In [2]:
!pip install xgboost -q

import pickle
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("Ready!")

Ready!


In [3]:
with open(DRIVE_PATH + 'processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train_sm = data['X_train_sm']
y_train_sm = data['y_train_sm']
X_test     = data['X_test']
y_test     = data['y_test']

print(f"Loaded. Train: {X_train_sm.shape}  Test: {X_test.shape}")

Loaded. Train: (501492, 225)  Test: (118108, 225)


In [5]:
# Logistic Regression
from sklearn.preprocessing import StandardScaler

print("Training Logistic Regression...")

# Scale the data first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)

# Increase max_iter to 2000
lr = LogisticRegression(max_iter=2000, random_state=42, solver='lbfgs')
lr.fit(X_train_scaled, y_train_sm)

print("Done!")

Training Logistic Regression...
Done!


In [6]:
# Random Forest
print("Training Random Forest... (5-10 mins)")

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)

print("Done!")

Training Random Forest... (5-10 mins)
Done!


In [7]:
# XGBoost
print("Training XGBoost... (best model, 5-10 mins)")

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,
    random_state=42,
    eval_metric='auc',
    use_label_encoder=False
)
xgb.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print("Done!")

Training XGBoost... (best model, 5-10 mins)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:38:00] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-auc:0.85041
[50]	validation_0-auc:0.88569
[100]	validation_0-auc:0.90291
[150]	validation_0-auc:0.91242
[200]	validation_0-auc:0.91976
[250]	validation_0-auc:0.92499
[299]	validation_0-auc:0.92995
Done!


In [8]:
# Save Models
with open(DRIVE_PATH + 'models.pkl', 'wb') as f:
    pickle.dump({
        'lr':  lr,
        'rf':  rf,
        'xgb': xgb
    }, f)

print("Models saved! Move to Notebook 4.")

Models saved! Move to Notebook 4.
